# Sponge Layer Visualization

This notebook implements and visualizes a sponge/buffer layer damping function for the cylindrical Boussinesq simulation.

## Theory

The sponge layer damps the solution near boundaries using:

$$D(r,z) = \frac{1 - T(z, L_{z,b}, s_z) \cdot T(r, L_{r,b}, s_r)}{\tau_b}$$

where $T(x, L, s)$ is a smooth transition function:

$$T(x, L, s) = \frac{1}{2}\left[\tanh\left(\frac{x+L}{s}\right) - \tanh\left(\frac{x-L}{s}\right)\right]$$

The damping is applied as:

$$\frac{d\psi}{dt} = \ldots - D(r,z) \psi$$

which gives:

$$\psi(t+\Delta t) = e^{-D(r,z)\Delta t} \psi(t)$$

## Notes

**Equation:**
- `D(r,z) = (1-T[z,Lz_b,sz]*T[r,Lr_b,sr])/tau_b`
- `d_t[psi(r,theta,z)] = ... - D(r,z)*psi(r,theta,z)`
- `T(x,L,s) = 0.5*(tanh[(x+L)/s]-tanh[(x-L)/s])`

**Parameters:**
- `ratio_L = 0.9`: Defines the active region (90% of domain)
- `ratio_s = 0.01`: Defines transition smoothness (1% of domain)
- `tau_b = 20`: Damping timescale in units of timesteps

**Boundary regions:**
- **Radial (one-sided)**: Active region `r[0]` to `r[ratio_L*Nr]`, sponge beyond
- **Axial (two-sided)**: Domain is `z in [0, ZLEN]`. Sponge at BOTH ends:
  - Lower boundary: near z=0
  - Upper boundary: near z=ZLEN

**Effective damping:**
- `psi' = exp(-D(r,z)*dt)*psi`
- With `ndt = tau_b`, we get `D'(r,z,dt) = exp(-(1-T[z]*T[r])/ndt)`

# Functions

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.io import FortranFile
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Dropdown, Checkbox, HBox, VBox, Label, interactive_output
import os

## Sponge function

In [2]:
def tanh_transition(x, L, s):
    """
    Smooth transition function using hyperbolic tangent.
    
    T(x, L, s) = 0.5 * [tanh((x+L)/s) - tanh((x-L)/s)]
    
    This function is ~1 inside [-L, L] and smoothly transitions to 0 outside.
    
    Parameters:
    -----------
    x : array_like
        Coordinate array
    L : float
        Half-width of the active region
    s : float
        Transition smoothness parameter (smaller = sharper transition)
    
    Returns:
    --------
    T : array_like
        Transition function values (0 to 1)
    """
    return 0.5 * (np.tanh((x + L) / s) - np.tanh((x - L) / s))


def compute_damping_function(r, z, tau_b, ratio_L=0.9, ratio_s=0.01):
    """
    Compute the sponge layer damping function D(r,z).
    
    D(r,z) = [1 - T(z, Lz_b, sz) * T(r, Lr_b, sr)] / tau_b
    
    Parameters:
    -----------
    r : array_like (1D)
        Radial grid points (from 0 to r_max)
    z : array_like (1D)
        Axial grid points (from 0 to ZLEN)
    tau_b : float
        Damping timescale (in time units)
    ratio_L : float
        Fraction of domain that is active (default 0.9)
    ratio_s : float
        Fraction of domain for transition smoothness (default 0.01)
    
    Returns:
    --------
    D : ndarray (2D)
        Damping function on (r,z) grid
    params : dict
        Dictionary with computed parameters for visualization
    """
    Nr = len(r)
    Nz = len(z)
    
    # Compute boundary parameters for radial direction (one-sided, damps at outer edge)
    idx_Lr = int(ratio_L * Nr)
    idx_sr = int((ratio_L + ratio_s) * Nr)
    if idx_sr >= Nr:
        idx_sr = Nr - 1
    # Lr_b = r[idx_Lr - 1]  # -1 for 0-based indexing
    # sr = r[idx_sr - 1] - Lr_b
    Lr_b = idx_Lr
    sr = idx_sr - Lr_b
    
    # Compute boundary parameters for axial direction (two-sided)
    # Domain: z in [z_min, z_max], need to damp at BOTH ends
    z_min = z[0]
    z_max = z[-1]
    z_len = z_max - z_min
    z_center = (z_min + z_max) / 2.0
    Lzh_b = (ratio_L * z_len) / 2.0 # Active region width = ratio_L * Nz * (z_len/Nz) = ratio_L * z_len
    sz = (ratio_s * z_len)
        
    # Compute transition functions
    T_r = tanh_transition(np.real(range(0,Nr)), Lr_b, sr)
    z_shifted = z - z_center
    T_z = tanh_transition(z_shifted, Lzh_b, sz)
    
    # Create 2D meshgrid
    R, Z = np.meshgrid(r, z, indexing='ij')
    T_R, T_Z = np.meshgrid(T_r, T_z, indexing='ij')    
    D = (1.0 - T_Z * T_R) / tau_b
    
    # Store parameters for visualization
    params = {
        'Lr_b': Lr_b,
        'sr': sr,
        'Lzh_b': Lzh_b,
        'sz': sz,
        'z_min': z_min,
        'z_max': z_max,
        'z_center': z_center,
        'z_active_min': z_center - Lzh_b,
        'z_active_max': z_center + Lzh_b,
        'T_r': T_r,
        'T_z': T_z,
        'idx_Lr': idx_Lr,
    }
    
    return D, params

def compute_effective_damping(D, dt):
    """
    Compute effective damping coefficient for exponential decay.
    
    D'(r,z,dt) = exp(-D(r,z) * dt)
    
    This represents the fraction of the solution remaining after one timestep.
    
    Parameters:
    -----------
    D : ndarray
        Damping function D(r,z)
    dt : float
        Timestep size
    
    Returns:
    --------
    D_eff : ndarray
        Effective damping coefficient (0 to 1)
    """
    return np.exp(-D * dt)

## Read Grid Data

In [3]:
def read_grid_from_output(directory):
    """
    Read radial and axial grid points from output directory.
    
    Reads r_colloc_pts.info and z_colloc_pts.info files.
    """
    r_file = os.path.join(directory, 'r_colloc_pts.info')
    z_file = os.path.join(directory, 'z_colloc_pts.info')
    
    if not os.path.exists(r_file) or not os.path.exists(z_file):
        print(f"Collocation point files not found in {directory}")
        if not os.path.exists(r_file):
            print(f"  Missing: r_colloc_pts.info")
        if not os.path.exists(z_file):
            print(f"  Missing: z_colloc_pts.info")
        print("Generating example grid instead...")
        return generate_example_grid()
    
    try:
        # Read radial collocation points
        print(f"Reading grid from: {directory}")
        r = np.loadtxt(r_file)
        z = np.loadtxt(z_file)
        
        print(f"Grid size: Nr={len(r)}, Nz={len(z)}")
        print(f"Radial range: r=[{r[0]:.4f}, {r[-1]:.4f}]")
        print(f"Axial range: z=[{z[0]:.4f}, {z[-1]:.4f}]")
        
        return r, z
    
    except Exception as e:
        print(f"Error reading files: {e}")

# Computation

In [4]:
# Read from directory
output_dir = '../../output/ab'
r, z = read_grid_from_output(output_dir)

# Default parameters
ratio_L_default = 0.90
ratio_s_default = 0.01
dt_default = 0.01  # Example timestep
tau_b_default = 20.0*dt_default

# Compute damping function with default parameters
D, params = compute_damping_function(r, z, tau_b_default, ratio_L_default, ratio_s_default)
D_eff = compute_effective_damping(D, dt_default)

print(f"\nDamping function parameters:")
print(f"  Radial boundary: r = 0 to {params['Lr_b']:.4f}")
print(f"  Radial smoothness: sr = {params['sr']:.4f}")
print(f"  Axial boundary: z = {params['z_active_min']:.4f} to {params['z_active_max']:.4f}")
print(f"  Axial smoothness: sz = {params['sz']:.4f}")
print(f"\nDamping function D(r,z):")
print(f"  Min: {D.min():.6f}")
print(f"  Max: {D.max():.6f}")
print(f"\nEffective damping exp(-D*dt) with dt={dt_default}:")
print(f"  Min: {D_eff.min():.6f} (strongest damping)")
print(f"  Max: {D_eff.max():.6f} (weakest damping)")

Reading grid from: ../../output/ab
Grid size: Nr=370, Nz=257
Radial range: r=[0.0130, 1232.5179]
Axial range: z=[0.0000, 10.4720]

Damping function parameters:
  Radial boundary: r = 0 to 333.0000
  Radial smoothness: sr = 3.0000
  Axial boundary: z = 0.5236 to 9.9484
  Axial smoothness: sz = 0.1047

Damping function D(r,z):
  Min: 0.000000
  Max: 5.000000

Effective damping exp(-D*dt) with dt=0.01:
  Min: 0.951229 (strongest damping)
  Max: 1.000000 (weakest damping)


# Visualizations

## Visualization Functions

In [5]:
def plot_damping_functions(tau_b, ratio_L, ratio_s, dt, plot_type='both', 
                          use_log=False, show_boundaries=True):
    """
    Plot damping functions D(r,z) and/or D'(r,z,dt).
    
    Parameters:
    -----------
    tau_b : float
        Damping timescale
    ratio_L : float
        Active region fraction (0.5 to 0.99)
    ratio_s : float
        Transition smoothness (0.001 to 0.1)
    dt : float
        Timestep size
    plot_type : str
        'D' for D(r,z), 'Deff' for D'(r,z,dt), 'both' for side-by-side
    use_log : bool
        Use logarithmic color scale
    show_boundaries : bool
        Mark sponge layer boundaries
    """
    # Compute damping functions
    D_plot, params_plot = compute_damping_function(r, z, tau_b, ratio_L, ratio_s)
    D_eff_plot = compute_effective_damping(D_plot, dt)
    
    # Create meshgrid for plotting
    R, Z = np.meshgrid(r, z, indexing='ij')
    
    # Determine subplot layout
    if plot_type == 'both':
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes_list = [axes[0], axes[1]]
    else:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        axes_list = [ax]
    
    # Plot D(r,z)
    if plot_type in ['D', 'both']:
        ax_idx = 0 if plot_type == 'both' else 0
        ax = axes_list[ax_idx]
        
        if use_log:
            # Use log scale with careful handling of zeros
            D_plot_pos = np.maximum(D_plot, 1e-10)
            norm = colors.LogNorm(vmin=D_plot_pos.min(), vmax=D_plot_pos.max())
            im = ax.contourf(R, Z, D_plot, levels=50, norm=norm, cmap='YlOrRd')
        else:
            im = ax.contourf(R, Z, D_plot, levels=50, cmap='YlOrRd')
        
        plt.colorbar(im, ax=ax, label=r'$D(r,z)$ [1/timestep]')
        ax.set_xlabel('Radial coordinate r')
        ax.set_ylabel('Axial coordinate z')
        ax.set_title(f'Damping Function $D(r,z)$\n' + 
                    r'$\tau_b=$' + f'{tau_b:.1f}, ' + 
                    r'$L_{ratio}=$' + f'{ratio_L:.2f}, ' +
                    r'$s_{ratio}=$' + f'{ratio_s:.3f}')
        # ax.set_xlim(r[0],r[params_plot['idx_Lr']]+10)
        ax.set_xlim(r[0],np.maximum(r[params_plot['idx_Lr']]+10,40))
        
        if show_boundaries:
            # Mark radial boundary
            ax.axvline(params_plot['Lr_b'], color='blue', linestyle='--', 
                      linewidth=1.5, label=f"$L_r$={params_plot['Lr_b']:.2f}")
            # Mark axial boundaries (both ends)
            ax.axhline(params_plot['z_active_min'], color='green', linestyle='--', 
                      linewidth=1.5, label=f"Active: zin[{params_plot['z_active_min']:.2f}, {params_plot['z_active_max']:.2f}]")
            ax.axhline(params_plot['z_active_max'], color='green', linestyle='--', linewidth=1.5)
            ax.legend(loc='upper right', fontsize=9)
            
    # Plot D'(r,z,dt)
    if plot_type in ['Deff', 'both']:
        ax_idx = 1 if plot_type == 'both' else 0
        ax = axes_list[ax_idx]
        
        if use_log:
            # For effective damping, use reversed log scale (1 - D_eff) to emphasize damping
            damping_strength = 1.0 - D_eff_plot
            damping_strength = np.maximum(damping_strength, 1e-10)
            norm = colors.LogNorm(vmin=damping_strength.min(), vmax=damping_strength.max())
            im = ax.contourf(R, Z, damping_strength, levels=50, norm=norm, cmap='YlOrRd')
            cbar_label = r'$1 - e^{-D \Delta t}$ (damping strength)'
        else:
            im = ax.contourf(R, Z, D_eff_plot, levels=50, cmap='RdYlGn')
            cbar_label = r"$D'(r,z,\Delta t) = e^{-D \Delta t}$ (retention fraction)"
        
        plt.colorbar(im, ax=ax, label=cbar_label)
        ax.set_xlabel('Radial coordinate r')
        ax.set_ylabel('Axial coordinate z')
        ax.set_title(f'Effective Damping after One Timestep\n' +
                    r'$\Delta t=$' + f'{dt:.4f}, ' +
                    r'$\tau_b=$' + f'{tau_b:.1f}')
        # ax.set_xlim(r[0],r[params_plot['idx_Lr']]+10)
        ax.set_xlim([r[0], np.maximum(r[params_plot['idx_Lr']]+10,40)])
        
        if show_boundaries:
            # Mark radial boundary
            ax.axvline(params_plot['Lr_b'], color='blue', linestyle='--', 
                      linewidth=1.5, label=f"$L_r$={params_plot['Lr_b']:.2f}")
            # Mark axial boundaries (both ends)
            ax.axhline(params_plot['z_active_min'], color='green', linestyle='--', 
                      linewidth=1.5, label=f"Active: zin[{params_plot['z_active_min']:.2f}, {params_plot['z_active_max']:.2f}]")
            ax.axhline(params_plot['z_active_max'], color='green', linestyle='--', linewidth=1.5)
            ax.legend(loc='upper right', fontsize=9)
            
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\n{'='*60}")
    print(f"Damping Function Statistics (tau_b={tau_b:.1f}, dt={dt:.4f})")
    print(f"{'='*60}")
    print(f"Active region: {ratio_L*100:.1f}% of domain")
    print(f"Transition width: {ratio_s*100:.1f}% of domain")
    print(f"\nDomain:")
    print(f"  z in [{params_plot['z_min']:.4f}, {params_plot['z_max']:.4f}]")
    print(f"\nBoundary positions:")
    print(f"  Radial: r < {params_plot['Lr_b']:.4f} (active), r > {params_plot['Lr_b']:.4f} (sponge)")
    print(f"  Axial: z in [{params_plot['z_active_min']:.4f}, {params_plot['z_active_max']:.4f}] (active)")
    print(f"         z < {params_plot['z_active_min']:.4f} (lower sponge)")
    print(f"         z > {params_plot['z_active_max']:.4f} (upper sponge)")
    print(f"\nD(r,z) range: [{D_plot.min():.6f}, {D_plot.max():.6f}]")
    print(f"Effective damping range: [{D_eff_plot.min():.6f}, {D_eff_plot.max():.6f}]")
    print(f"  (Values close to 1 mean weak damping, close to 0 mean strong damping)")
    
    # Compute e-folding time in sponge regions
    max_damping_rate = D_plot.max()
    if max_damping_rate > 0:
        e_folding_steps = 1.0 / (max_damping_rate * dt)
        print(f"\nMaximum damping region:")
        print(f"  e-folding time: {e_folding_steps:.1f} timesteps")
        print(f"  After {tau_b:.4f} time units: amplitude reduced by factor of {np.exp(-max_damping_rate * dt * tau_b):.6f}")

def plot_1d_profiles(tau_b, ratio_L, ratio_s):
    """
    Plot 1D profiles of transition functions T(r) and T(z).
    """
    # Compute transition functions
    D_plot, params_plot = compute_damping_function(r, z, tau_b, ratio_L, ratio_s)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Plot T(r)
    axes[0].plot(r, params_plot['T_r'], 'b-', linewidth=2)
    axes[0].axvline(params_plot['Lr_b'], color='red', linestyle='--', 
                   label=f"$L_r$={params_plot['Lr_b']:.2f}")
    axes[0].axvline(params_plot['Lr_b'] + params_plot['sr'], color='orange', 
                   linestyle=':', label=f"$L_r + s_r$={params_plot['Lr_b']+params_plot['sr']:.2f}")
    axes[0].set_xlabel('Radial coordinate r')
    axes[0].set_ylabel(r'$T(r, L_r, s_r)$')
    axes[0].set_title('Radial Transition Function')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[0].set_ylim([-0.1, 1.1])
    # axes[0].set_xlim([r[0], r[params_plot['idx_Lr']]+10])
    axes[0].set_xlim([r[0], np.maximum(r[params_plot['idx_Lr']]+10,40)])
    # axes[0].plot(range(0,len(r)), params_plot['T_r'], 'b-', linewidth=2)
    # axes[0].axvline(params_plot['Lr_b'], color='red', linestyle='--', 
    #                label=f"$L_r$={params_plot['Lr_b']:.2f}")
    # axes[0].axvline(params_plot['Lr_b'] + params_plot['sr'], color='orange', 
    #                linestyle=':', label=f"$L_r + s_r$={params_plot['Lr_b']+params_plot['sr']:.2f}")
    # axes[0].set_xlabel('Radial index r#')
    # axes[0].set_ylabel(r'$T(r, L_r, s_r)$')
    # axes[0].set_title('Radial Transition Function')
    # axes[0].grid(True, alpha=0.3)
    # axes[0].legend()
    # axes[0].set_ylim([-0.1, 1.1])
    # axes[0].set_xlim([1, len(r)])
    
    # Plot T(z)
    axes[1].plot(z, params_plot['T_z'], 'g-', linewidth=2)
    # Mark active region boundaries
    axes[1].axvline(params_plot['z_active_min'], color='red', linestyle='--', 
                   label=f"Active region")
    axes[1].axvline(params_plot['z_active_max'], color='red', linestyle='--')
    # Mark transition region boundaries
    z_trans_lower = params_plot['z_active_min'] - params_plot['sz']
    z_trans_upper = params_plot['z_active_max'] + params_plot['sz']
    axes[1].axvline(z_trans_lower, color='orange', linestyle=':', 
                   label=f"Transition region")
    axes[1].axvline(z_trans_upper, color='orange', linestyle=':')
    axes[1].set_xlabel('Axial coordinate z')
    axes[1].set_ylabel(r'$T(z)$')
    axes[1].set_title('Axial Transition Function (Two-Sided)')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[1].set_ylim([-0.1, 1.1])
    
    plt.tight_layout()
    plt.show()
    
    print(f"Transition function properties:")
    print(f"  Radial: Active (T≈1) for r < {params_plot['Lr_b']:.4f}")
    print(f"          Transition region: r in [{params_plot['Lr_b']:.4f}, {params_plot['Lr_b']+params_plot['sr']:.4f}]")
    print(f"  Axial:  Active (T≈1) for z in [{params_plot['z_active_min']:.4f}, {params_plot['z_active_max']:.4f}]")
    print(f"          Lower transition: z in [{z_trans_lower:.4f}, {params_plot['z_active_min']:.4f}]")
    print(f"          Upper transition: z in [{params_plot['z_active_max']:.4f}, {z_trans_upper:.4f}]")
    print(f"          Full sponge: z < {z_trans_lower:.4f} or z > {z_trans_upper:.4f}")


## Results

In [6]:
# Create interactive widgets
tau_b_slider = FloatSlider(
    value=20.0*dt_default, min=dt_default, max=100.0*dt_default, step=1.0*dt_default,
    description=r'$\tau_b$:',
    continuous_update=False,
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

ratio_L_slider = FloatSlider(
    value=0.90, min=0.50, max=0.99, step=0.01,
    description=r'$L_{ratio}$:',
    continuous_update=False,
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

ratio_s_slider = FloatSlider(
    value=0.01, min=0.001, max=0.1, step=0.001,
    description=r'$s_{ratio}$:',
    continuous_update=False,
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

dt_slider = FloatSlider(
    value=0.01, min=0.001, max=0.1, step=0.001,
    description=r'$\Delta t$:',
    continuous_update=False,
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

plot_type_dropdown = Dropdown(
    options=[('Both D and D\'', 'both'), ('D(r,z) only', 'D'), ('D\'(r,z,dt) only', 'Deff')],
    value='both',
    description='Plot:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

log_scale_check = Checkbox(
    value=False,
    description='Logarithmic color scale',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)

boundaries_check = Checkbox(
    value=True,
    description='Show boundary lines',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)

# Organize widgets in compact layout
ui = VBox([
    Label('Sponge Layer Parameters:'),
    HBox([tau_b_slider, ratio_L_slider]),
    HBox([ratio_s_slider, dt_slider]),
    Label('Visualization Options:'),
    HBox([plot_type_dropdown]),
    HBox([log_scale_check, boundaries_check])
])

# Create interactive output
out = interactive_output(
    plot_damping_functions,
    {
        'tau_b': tau_b_slider,
        'ratio_L': ratio_L_slider,
        'ratio_s': ratio_s_slider,
        'dt': dt_slider,
        'plot_type': plot_type_dropdown,
        'use_log': log_scale_check,
        'show_boundaries': boundaries_check
    }
)

# Display
display(ui, out)

Output()

In [ ]:
# Interactive 1D profile plot
interact(plot_1d_profiles,
         tau_b=FloatSlider(value=20.0*dt_default, min=1.0*dt_default, max=100.0*dt_default, step=1.0*dt_default,
                          description=r'$\tau_b$:', continuous_update=False),
         ratio_L=FloatSlider(value=0.90, min=0.50, max=0.99, step=0.01,
                            description=r'$L_{ratio}$:', continuous_update=False),
         ratio_s=FloatSlider(value=0.01, min=0.001, max=0.1, step=0.001,
                            description=r'$s_{ratio}$:', continuous_update=False));

interactive(children=(FloatSlider(value=0.2, continuous_update=False, description='$\\tau_b$:', max=1.0, min=0…